# 筛选全程存在的传感器

统计所有元数据文件中均出现的传感器ID，保存其元数据。

In [1]:
import pandas as pd
import glob
import os
import re
from datetime import datetime

In [2]:
# 配置
META_FOLDER = "../d03_meta"  # 修改为你的路径
OUTPUT_DIR = "../d03_meta_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = datetime(2024, 11, 1)
END_DATE = datetime(2025, 12, 31)

In [3]:
def parse_date(filename):
    match = re.search(r'(\d{4})_(\d{2})_(\d{2})', os.path.basename(filename))
    if match:
        return datetime(int(match.group(1)), int(match.group(2)), int(match.group(3)))
    return None

# 获取2025年文件
all_files = sorted(glob.glob(os.path.join(META_FOLDER, "d*_text_meta_*.txt")))
files_2025 = [(f, parse_date(f)) for f in all_files if parse_date(f) and START_DATE <= parse_date(f) <= END_DATE]
files_2025.sort(key=lambda x: x[1])

print(f"2025年元数据文件数: {len(files_2025)}")

2025年元数据文件数: 36


In [4]:
# 找出所有文件中均存在的传感器ID（取交集）
common_ids = None

for file_path, date in files_2025:
    df = pd.read_csv(file_path, sep='\t', usecols=[0], dtype=str)
    ids = set(df.iloc[:, 0].dropna())
    
    if common_ids is None:
        common_ids = ids
    else:
        common_ids = common_ids & ids  # 交集

print(f"所有文件中均存在的传感器数: {len(common_ids)}")

所有文件中均存在的传感器数: 1830


In [5]:
# 从最新的元数据文件中提取这些传感器的完整信息
latest_file = files_2025[-1][0]
print(f"使用最新元数据: {os.path.basename(latest_file)}")

columns = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

meta_df = pd.read_csv(
    latest_file, sep='\t', names=columns, header=0,
    dtype={'ID': str, 'Fwy': str, 'Dir': str, 'Type': str}
)

# 筛选全程存在的传感器
common_meta = meta_df[meta_df['ID'].isin(common_ids)].copy()
print(f"筛选后传感器数: {len(common_meta)}")

使用最新元数据: d03_text_meta_2025_12_30.txt
筛选后传感器数: 1830


In [6]:
# 统计
print("\n按类型统计:")
print(common_meta['Type'].value_counts())

print("\n按高速公路统计 (前10):")
print(common_meta['Fwy'].value_counts().head(10))


按类型统计:
Type
ML    854
OR    413
HV    269
FR    266
FF     28
Name: count, dtype: int64

按高速公路统计 (前10):
Fwy
80     458
50     455
99     326
5      271
51      90
65      65
20      40
70      24
113     19
89      16
Name: count, dtype: int64


In [7]:
# 预览
print("\n数据预览:")
display(common_meta[['ID', 'Fwy', 'Dir', 'Type', 'Abs_PM', 'Lanes', 'Name']].head(20))


数据预览:


,ID,Fwy,Dir,Type,Abs_PM,Lanes,Name
0,308511,50,E,ML,60.162,2,Sly Park Rd
1,308512,50,W,ML,60.166,2,Sly Park Rd
2,311831,5,S,OR,506.189,1,Elk Grove Blvd to 5SB Loop
3,311832,5,S,FR,506.189,1,5SB to Elk Grove Blvd
4,311844,5,N,OR,506.373,2,Elk Grove Blvd 5NB Slip
5,311847,5,N,OR,507.478,3,Laguna Blvd to 5NB Slip
6,311864,5,N,FR,507.226,1,5NB to Laguna Blvd
7,311903,50,E,ML,3.789,3,50EB at 6TH Street
8,311930,50,E,FF,3.788,3,5NB and 5SB to 50EB
9,311973,50,E,OR,4.376,1,13th St


In [8]:
# 保存
output_path = os.path.join(OUTPUT_DIR, 'sensors_common_2025.csv')
common_meta.to_csv(output_path, index=False)
print(f"\n已保存: {output_path}")
print(f"共 {len(common_meta)} 个传感器")


已保存: ../d03_meta_processed/sensors_common_2025.csv
共 1830 个传感器
